<!-- NOTEBOOK_HEADER: This notebook is automatically generated by tools/build_notebook.py -->


# Kaggle training, verification, and evaluation

For the first run, execute the config, clone/install, CLI-contract, and verification cells manually and read the verification output before continuing.

`SESSION_PHASE` defaults to `setup_cache` so the first notebook pass always clones, verifies, restores HF state, and builds the cache; later sessions can switch to `train`, `eval`, or `all` when continuing work.

`train` runs all branch training; `eval` runs evaluation/results; `all` runs every phase. Every phase restores and syncs the same HF paths.


In [ ]:
# 1. Config
import os, re, sys, json, time, subprocess, threading, csv
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/Algorithm-bot/ai-vs-real-face-detector.git"
GITHUB_BRANCH = "hybrid-detector-eval-overhaul"
KAGGLE_DATASET_MOUNT = Path("/kaggle/input/datasets/sahilsawantspit/ai-vs-real-face-detector-data")
SESSION_PHASE = "setup_cache"  # setup_cache | train | eval | all
VERIFY_STRICT = True
FORCE_RETRAIN = False
SEEDS = [42]
SUBSET_SEED = 42
MAX_TRAIN_IMAGES = 200_000
SEMANTIC_MODEL = "vit_large_patch14_clip_224.openai"
SEMANTIC_DIM = 768  # SemanticEncoder documents CLIP ViT-L/14's projected output width.
CACHE_BRANCHES = ["semantic", "physics", "freq"]
CACHE_BRANCHES_ARG = ",".join(CACHE_BRANCHES)
PREPROCESS = "crop"
CACHE_AUGMENT = True
NUM_WORKERS = 3
SYNC_INTERVAL_S = 300
BRANCHES = ["semantic_only", "physics_only", "freq_only", "stage1", "full_hybrid"]
HPARAMS = {
    "semantic_only": dict(epochs=60, batch_size=128, lr=1e-3),
    "physics_only": dict(epochs=60, batch_size=128, lr=1e-3),
    "freq_only": dict(epochs=60, batch_size=128, lr=1e-3),
    "stage1": dict(epochs=4, batch_size=32, lr=1e-4),
    "full_hybrid": dict(epochs=4, batch_size=32, lr=1e-4),
}
HF_CHECKPOINT_REPO = "sahilSpit/checkpoints"
HF_CHECKPOINT_PRIVATE = True
WORKING = Path("/kaggle/working")
MODELS_ROOT = WORKING / "models_v2"
FEATURE_CACHE_DIR = WORKING / "feature_cache_v2"
EVAL_ROOT = WORKING / "evaluation_v2"
log_path = WORKING / "train.log"
assert SESSION_PHASE in {"all", "setup_cache", "train", "eval"}
assert not [mode for mode in BRANCHES if mode not in HPARAMS]
print(f"phase={SESSION_PHASE}; seeds={SEEDS}; cache branches={CACHE_BRANCHES_ARG}")


In [ ]:
# 2. Clone and install (always run)
import shutil
clone_dir = WORKING / "ai-vs-real-face-detector"
if clone_dir.exists():
    shutil.rmtree(clone_dir)
subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(clone_dir)], check=True)
project_root = clone_dir
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(project_root / "requirements.txt")], check=True)
sys.path.insert(0, str(project_root))
print("Project root:", project_root)


In [ ]:
# 2a. Assert every notebook flag exists in the real argparse help.
def has_flag(help_text, flag):
    return re.search(rf"(?<![A-Za-z0-9_-]){re.escape(flag)}(?![A-Za-z0-9_-])", help_text) is not None

help_contract = {
    "src/cache_features.py": ["--data-dir", "--cache-dir", "--workers", "--semantic-model", "--preprocess", "--augment", "--batch-size", "--max-train-images", "--branches", "--seed", "--subset-seed"],
    "src/train.py": ["--mode", "--data-dir", "--output-dir", "--epochs", "--batch-size", "--lr", "--num-workers", "--max-train-images", "--seed", "--subset-seed", "--feature-cache", "--branches", "--branch-dropout", "--fusion-mode", "--semantic-model", "--semantic-dim", "--preprocess"],
    "src/train_cached.py": ["--mode", "--data-dir", "--output-dir", "--epochs", "--batch-size", "--lr", "--feature-cache", "--max-train-images", "--seed", "--subset-seed"],
    "src/evaluate.py": ["--checkpoint", "--data-dir", "--output-dir", "--feature-cache", "--splits-yaml", "--eval-set", "--n-bootstrap", "--seed"],
    "src/calibrate_threshold.py": ["--checkpoint", "--data-dir", "--feature-cache", "--splits-yaml"],
    "src/stack_fusion.py": ["--predictions", "--splits-yaml", "--output-dir", "--n-bootstrap", "--seed"],
    "src/diagnose_branches.py": ["--feature-cache", "--predictions", "--splits-yaml", "--output"],
    "kaggle_verify.py": ["--data-dir", "--feature-cache"],
}
from concurrent.futures import ThreadPoolExecutor
def _read_help(item):
    relative, flags = item
    env = {**os.environ, "BRANCH_PROJECT_ROOT": str(project_root), "BRANCH_FEATURE_CACHE": str(FEATURE_CACHE_DIR)}
    result = subprocess.run([sys.executable, str(project_root / relative), "--help"], text=True, capture_output=True, env=env)
    text = result.stdout + result.stderr
    missing = [flag for flag in flags if not has_flag(text, flag)]
    return relative, {"returncode": result.returncode, "missing": missing}
with ThreadPoolExecutor(max_workers=len(help_contract)) as pool:
    heat_results = dict(pool.map(_read_help, help_contract.items()))
for relative, result in heat_results.items():
    print(relative, result)
assert all(row["returncode"] == 0 and not row["missing"] for row in heat_results.values()), heat_results
print("CLI help contract: OK")


In [ ]:
# 2b. Verification gate (setup_cache only)
if SESSION_PHASE in {"all", "setup_cache"}:
    verify_cmd = [sys.executable, str(project_root / "kaggle_verify.py"), "--data-dir", str(KAGGLE_DATASET_MOUNT), "--feature-cache", str(FEATURE_CACHE_DIR)]
    verify_rc = subprocess.run(verify_cmd).returncode
    print("kaggle_verify.py return code:", verify_rc)
    if VERIFY_STRICT:
        assert verify_rc == 0, "Verification gate failed; read its complete report before continuing."
else:
    print(f"Skipping verification for phase {SESSION_PHASE}")


In [ ]:
# 3. Restore and sync HF artifacts (all phases)
from huggingface_hub import HfApi, snapshot_download
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
api.create_repo(repo_id=HF_CHECKPOINT_REPO, repo_type="model", private=HF_CHECKPOINT_PRIVATE, exist_ok=True)
allow_patterns = ["feature_cache_v2/*", "evaluation_v2/*", "logs/*"] + [f"models_v2/{mode}/seed_{seed}/*" for mode in BRANCHES for seed in SEEDS]
# Attempt to restore prior artifacts from HF
try:
    snapshot_download(repo_id=HF_CHECKPOINT_REPO, repo_type="model", token=hf_token, local_dir=str(WORKING), allow_patterns=allow_patterns)
except Exception as exc:
    print("No prior artifacts restored:", exc)
SYNC_DIRS = {FEATURE_CACHE_DIR: "feature_cache_v2", EVAL_ROOT: "evaluation_v2"}
for mode in BRANCHES:
    for seed in SEEDS:
        SYNC_DIRS[MODELS_ROOT / mode / f"seed_{seed}"] = f"models_v2/{mode}/seed_{seed}"
uploaded = {}

def sync_once(min_age=60):
    for local_dir, remote_dir in SYNC_DIRS.items():
        if not local_dir.exists():
            continue
        for path in local_dir.rglob("*"):
            if not path.is_file():
                continue
            st = path.stat()
            if time.time() - st.st_mtime < min_age:
                continue
            if uploaded.get(str(path)) == st.st_mtime:
                continue
            try:
                api.upload_file(
                    path_or_fileobj=str(path),
                    path_in_repo=f"{remote_dir}/{path.relative_to(local_dir).as_posix()}",
                    repo_id=HF_CHECKPOINT_REPO,
                    repo_type="model",
                )
                uploaded[str(path)] = st.st_mtime
            except Exception as exc:
                print(f"WARN: sync failed for {path}: {exc}")

stop_sync = threading.Event()
def _sync_loop():
    while not stop_sync.is_set():
        sync_once()
        stop_sync.wait(SYNC_INTERVAL_S)
threading.Thread(target=_sync_loop, daemon=True).start()


In [ ]:
# 4. Build/reuse cache (setup_cache only)
if SESSION_PHASE in {"all", "setup_cache"}:
    cache_cmd = [sys.executable, str(project_root / "src/cache_features.py"), "--data-dir", str(KAGGLE_DATASET_MOUNT), "--cache-dir", str(FEATURE_CACHE_DIR), "--workers", str(NUM_WORKERS), "--semantic-model", SEMANTIC_MODEL, "--preprocess", PREPROCESS, "--batch-size", "64", "--max-train-images", str(MAX_TRAIN_IMAGES), "--branches", CACHE_BRANCHES_ARG, "--seed", str(SEEDS[0]), "--subset-seed", str(SUBSET_SEED)]
    if CACHE_AUGMENT:
        cache_cmd.append("--augment")
    assert subprocess.run(cache_cmd, env={**os.environ, "HF_TOKEN": hf_token, "PYTHONUNBUFFERED": "1"}).returncode == 0
    from src.eval.cache_meta import load_cache_meta, normalise_branches
    for split in ("train", "val", "test"):
        assert (FEATURE_CACHE_DIR / f"{split}.npz").exists(), f"missing {split}.npz"
    meta = load_cache_meta(FEATURE_CACHE_DIR)
    assert normalise_branches(meta.get("branches")) == sorted(CACHE_BRANCHES)
    assert meta.get("max_train_images") == MAX_TRAIN_IMAGES
    assert meta.get("semantic_model") == SEMANTIC_MODEL
    assert meta.get("semantic_dim") == SEMANTIC_DIM
    assert meta.get("preprocess") == PREPROCESS
    assert bool(meta.get("augment")) is CACHE_AUGMENT
    print("cache metadata contract: OK")

    # Sync and verify cache after build
    def sync_and_verify_cache(max_attempts=3):
        for attempt in range(1, max_attempts + 1):
            sync_once(min_age=0)
            remote = set(api.list_repo_files(repo_id=HF_CHECKPOINT_REPO, repo_type="model"))
            missing = [
                f"feature_cache_v2/{p.relative_to(FEATURE_CACHE_DIR).as_posix()}"
                for p in FEATURE_CACHE_DIR.rglob("*") if p.is_file()
                and f"feature_cache_v2/{p.relative_to(FEATURE_CACHE_DIR).as_posix()}" not in remote
            ]
            if not missing:
                print("Cache fully on HF.")
                return
            print(f"Attempt {attempt}/{max_attempts}: {len(missing)} files not on HF yet, e.g. {missing[:3]}")
            if attempt < max_attempts:
                time.sleep(10)
        raise RuntimeError(f"Cache not fully on HF: {missing[:10]}")
    sync_and_verify_cache()
else:
    print(f"Skipping cache build for phase {SESSION_PHASE}")


In [ ]:
# 5. Train every mode and seed (train only). Stage1 deliberately has no --feature-cache.
def run_and_log(cmd, extra_env=None):
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "HF_TOKEN": hf_token, **(extra_env or {})}
    state = {"last_bar": 0.0}
    def emit(raw, log):
        line = raw.decode("utf-8", errors="replace").rstrip()
        if not line:
            return
        bar = "it/s]" in line or "s/it]" in line or "%|" in line
        if bar and "100%|" not in line:
            if time.time() - state["last_bar"] < 30:
                return
            state["last_bar"] = time.time()
        print(line, flush=True)
        log.write(line)
        log.write("
")
        log.flush()
    with open(log_path, "a", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
        buffer = b""
        while chunk := proc.stdout.read1(4096):
            buffer += chunk
            # Split on line endings and remove them
            parts = buffer.splitlines()
            for raw in parts:
                emit(raw, log)
        proc.wait()
    return proc.returncode

def build_cmd(mode, seed):
    hp, out = HPARAMS[mode], MODELS_ROOT / mode / f"seed_{seed}"
    common = ["--mode", mode, "--data-dir", str(KAGGLE_DATASET_MOUNT), "--output-dir", str(out), "--epochs", str(hp["epochs"]), "--batch-size", str(hp["batch_size"]), "--lr", str(hp["lr"]), "--seed", str(seed), "--max-train-images", str(MAX_TRAIN_IMAGES), "--subset-seed", str(SUBSET_SEED)]
    if mode in {"semantic_only", "physics_only", "freq_only"}:
        return [sys.executable, "-u", str(project_root / "src/train_cached.py"), *common, "--feature-cache", str(FEATURE_CACHE_DIR), "--semantic-model", SEMANTIC_MODEL, "--semantic-dim", str(SEMANTIC_DIM), "--preprocess", PREPROCESS, "--branches", CACHE_BRANCHES_ARG], {"BRANCH_PROJECT_ROOT": str(project_root), "BRANCH_FEATURE_CACHE": str(FEATURE_CACHE_DIR)}
    cmd = [sys.executable, "-u", str(project_root / "src/train.py"), *common, "--data-source", "local", "--num-workers", str(NUM_WORKERS), "--preprocess", PREPROCESS]
    if mode == "full_hybrid":
        cmd += ["--feature-cache", str(FEATURE_CACHE_DIR), "--semantic-model", SEMANTIC_MODEL, "--semantic-dim", str(SEMANTIC_DIM), "--branches", "deep,physics,semantic,freq", "--branch-dropout", "0.15", "--fusion-mode", "gated"]
    return cmd, {}

# Periodic log upload thread
def _log_loop():
    while not stop_sync.is_set():
        if log_path.exists():
            try:
                api.upload_file(path_or_fileobj=str(log_path), path_in_repo="logs/train.log", repo_id=HF_CHECKPOINT_REPO, repo_type="model")
            except Exception as exc:
                print("WARN: log sync failed:", exc)
        stop_sync.wait(120)  # Every 2 minutes
threading.Thread(target=_log_loop, daemon=True).start()

if SESSION_PHASE in {"all", "train"}:
    skipped = []
    for seed in SEEDS:
        for mode in BRANCHES:
            out = MODELS_ROOT / mode / f"seed_{seed}"
            if (out / f"{mode}_history.json").exists() and not FORCE_RETRAIN:
                skipped.append(f"{mode}/seed_{seed}")
                continue
            cmd, env = build_cmd(mode, seed)
            assert ("--feature-cache" in cmd) == (mode != "stage1"), cmd
            rc = run_and_log(cmd, env); sync_once(min_age=0)
            if rc:
                raise RuntimeError(f"{mode}/seed_{seed} failed: {rc}")
    print("Skipped (already finished):", skipped or "none")
else:
    print(f"Skipping training for phase {SESSION_PHASE}")


In [ ]:
# 6. Evaluate, calibrate, stack, and diagnose (eval only); fail on any subprocess error.
if SESSION_PHASE in {"all", "eval"}:
    import pandas as pd
    from src.eval.splits import load_split_spec
    from src.eval.tables import build_eval_tables
    failures, evaluation_rows = {}, []
    for seed in SEEDS:
        for mode in BRANCHES:
            root = MODELS_ROOT / mode / f"seed_{seed}"; checkpoint = root / f"{mode}_best.pt"; output = EVAL_ROOT / mode / f"seed_{seed}"
            eval_cmd = [sys.executable, str(project_root / "src/evaluate.py"), "--checkpoint", str(checkpoint), "--data-dir", str(KAGGLE_DATASET_MOUNT), "--output-dir", str(output), "--feature-cache", str(FEATURE_CACHE_DIR), "--splits-yaml", str(project_root / "eval/splits.yaml"), "--eval-set", "all", "--n-bootstrap", "1000", "--seed", str(seed)]
            cal_cmd = [sys.executable, str(project_root / "src/calibrate_threshold.py"), "--checkpoint", str(checkpoint), "--data-dir", str(KAGGLE_DATASET_MOUNT), "--feature-cache", str(FEATURE_CACHE_DIR), "--splits-yaml", str(project_root / "eval/splits.yaml")]
            evaluation_rows.append({"mode": mode, "seed": seed, "eval ok": run_and_log(eval_cmd) == 0, "calibrate ok": run_and_log(cal_cmd) == 0, "stack ok": None, "diagnose ok": None})
    by_seed = {seed: [] for seed in SEEDS}
    for row in evaluation_rows:
        prediction = EVAL_ROOT / row["mode"] / f"seed_{row['seed']}" / "predictions.csv"
        if prediction.exists(): by_seed[row["seed"]].append(prediction)
    for seed, paths in by_seed.items():
        merged = None
        for path in paths:
            df = pd.read_csv(path); merged = df if merged is None else merged.merge(df.drop(columns=[c for c in ["folder", "source", "label", "p_ai", "logit_real", "logit_ai", "split"] if c in df]), on="path", how="outer")
        combined = EVAL_ROOT / f"combined_seed_{seed}.csv"; merged.to_csv(combined, index=False)
        stack_cmd = [sys.executable, str(project_root / "src/stack_fusion.py"), "--predictions", str(combined), "--splits-yaml", str(project_root / "eval/splits.yaml"), "--output-dir", str(EVAL_ROOT / "stacking" / f"seed_{seed}"), "--seed", str(seed)]
        diag_cmd = [sys.executable, str(project_root / "src/diagnose_branches.py"), "--predictions", str(combined), "--splits-yaml", str(project_root / "eval/splits.yaml"), "--output", str(EVAL_ROOT / f"diagnostics_seed_{seed}.csv")]
        stack_ok, diag_ok = run_and_log(stack_cmd) == 0, run_and_log(diag_cmd) == 0
        for row in evaluation_rows:
            if row["seed"] == seed: row["stack ok"], row["diagnose ok"] = stack_ok, diag_ok
    summary = pd.DataFrame(evaluation_rows); print(summary.to_string(index=False))

    # Write summary table to evaluation_v2/ as JSON and CSV
    summary_csv_path = EVAL_ROOT / "summary.csv"
    summary_json_path = EVAL_ROOT / "summary.json"
    summary.to_csv(summary_csv_path, index=False)
    summary.to_json(summary_json_path, indent=2)
    print(f"Summary table written to {summary_csv_path} and {summary_json_path}")

    if not summary[["eval ok", "calibrate ok", "stack ok", "diagnose ok"]].fillna(False).all().all(): raise RuntimeError("Evaluation subprocess failure(s); see summary above.")
else:
    print(f"Skipping evaluation for phase {SESSION_PHASE}")


In [ ]:
# 7. Per-seed fold table and multi-seed mean +/- std.
if SESSION_PHASE in {"all", "eval"}:
    from src.eval.splits import load_split_spec
    from src.eval.tables import build_eval_tables
    spec, rows = load_split_spec(project_root / "eval/splits.yaml"), []
    for seed in SEEDS:
        df = pd.read_csv(EVAL_ROOT / f"combined_seed_{seed}.csv")
        for name, folders in {"fold_a": spec.dev, "fold_b": spec.report, "progan": spec.progan}.items():
            part = df[df.floor.map(lambda f: spec.assign(str(f)) == ({"fold_a":"dev", "fold_b":"report", "progan":"progan"}[name]))]
            tables = build_eval_tables(part.to_dict("records"), n_bootstrap=1000, seed=seed)
            rows.append({"seed": seed, "split": name, "metrics": tables["at_0.5"]["overall"]})
    comparison = {r["split"]: {} for r in rows}
    for r in rows: comparison[r["split"]][f"seed_{r['seed']}"] = r["metrics"]
    required = {"fold_a", "fold_b", "progan"}; found = set(comparison)
    print("Found result keys:", sorted(found)); assert required <= found, f"Missing result keys: {required - found}"
    (EVAL_ROOT / "branch_comparison_v2.json").write_text(json.dumps(comparison, indent=2, default=str))
    for split in sorted(required):
        aucs = [comparison[split][f"seed_{seed}"].get("roc_auc") for seed in SEEDS]
        print(f"{split}: AUC mean +/- std = {sum(aucs)/len(aucs):.4f} +/- {pd.Series(aucs).std(ddof=0):.4f}")
    if len(SEEDS) == 1: print("One seed: bootstrap CIs are image-level resampling only, not training variance.")


# Notes

Current branches: `semantic_only`, `physics_only`, `freq_only`, `stage1`, `full_hybrid`. Checkpoints are at `models_v2/<mode>/seed_<seed>/`; the comparison JSON is `evaluation_v2/branch_comparison_v2.json`.

To retrain one branch, set `BRANCHES = ["freq_only"]` (or another current mode) and run the train phase. Pull a checkpoint with `hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename="models_v2/freq_only/seed_42/freq_only_best.pt")".
